# Week 6 — Stop 3: Production RAG System

**M2 Capstone — Graded Submission**

This notebook demonstrates the three production-grade capabilities added in Stop 3:

| Component | Description |
|-----------|-------------|
| **1. Graph RAG** | Entity triples extracted from corpus → knowledge graph → 1-2 hop subgraph traversal |
| **2. Hallucination Guardrails** | Citation binding per sentence → atomic claim verification → decision gate |
| **3. Multimodal Retrieval** | Text (vector store) + Tables (CSV) + Images (caption-based) with per-item citation tags |

All three modalities use late fusion, deduplicated evidence keys, and per-query observability logging.

---

**Corpus:** TechStore Plus — product manuals, support articles, warranty/return policies, laptop specs CSVs, product images.

**Prerequisite:** Stop 2 pipeline working (MMR + cross-encoder). Run `pytest tests/test_stop2_vectorstore.py` to confirm.

## Cell 1 — Setup

In [ ]:
import os, sys
from pathlib import Path
import logging

# Set working directory to the capstone root so relative imports work
CAPSTONE_DIR = Path("c03-t05-bruno-pieri-m2-challenge").resolve()
os.chdir(CAPSTONE_DIR)
if str(CAPSTONE_DIR) not in sys.path:
    sys.path.insert(0, str(CAPSTONE_DIR))

from dotenv import load_dotenv
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set — check .env"

# Enable RAG observability logs
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

print(f"CWD: {Path.cwd()}")
print("Setup complete.")

---
## Component 1 — Graph RAG Layer

Extracts `(subject, relation, object)` triples from corpus documents via LLM,
builds a directed NetworkX property graph, and traverses it (BFS, 1-2 hops)
to surface entity relationships that vector search would miss.

In [ ]:
from src.pipeline.loader import load_documents
from src.graph.knowledge_graph import TechStoreKnowledgeGraph

print("Loading corpus documents...")
docs = load_documents()
print(f"Loaded {len(docs)} documents.")

print("\nBuilding knowledge graph (LLM triple extraction)...")
kg = TechStoreKnowledgeGraph()
kg.extract_and_build(docs)
print(f"Graph: {kg.graph.number_of_nodes()} nodes, {kg.graph.number_of_edges()} edges")

In [ ]:
import networkx as nx

# Show all extracted triples
print("Extracted triples (subject → relation → object):")
print("-" * 80)
for u, v, data in sorted(kg.graph.edges(data=True), key=lambda x: x[2].get('relation','')):
    print(f"  {u:30s} --{data.get('relation','?'):15s}--> {v}")
    print(f"    source: {data.get('source_id','?')}")
    print(f"    quote:  {data.get('quote','')[:80]}")
    print()

In [ ]:
# Demonstrate 1-2 hop subgraph traversal
print("=== Subgraph Query: seed=['laptop pro x1'] (2 hops) ===")
snippets = kg.query_subgraph(["Laptop Pro X1"], hops=2)
for s in snippets:
    print(f"  hop={s['hop']}  {s['subject']} --{s['relation']}--> {s['object']}")
    print(f"    [{s['source_id']}] {s['quote'][:80]}")

print(f"\nTotal snippets returned: {len(snippets)}")

In [ ]:
# Multi-hop lineage query: which products are covered by extended warranty?
print("=== Multi-hop: 'extended warranty' coverage chain ===")
snippets_w = kg.query_subgraph(["extended warranty", "Premium Protection Plan"], hops=2)
for s in snippets_w:
    print(f"  hop={s['hop']}  {s['subject']} --{s['relation']}--> {s['object']}")

---
## Component 2 — Hallucination Guardrails

Every generated sentence ends with a bracketed citation key `[source]`. 
The verifier decomposes the answer into atomic claims, checks each against
the cited context via entailment, and routes to one of four outcomes:
`answer` | `answer_with_disclaimer` | `extractive` | `no_answer`.

In [ ]:
from src.pipeline.vectorstore import load_vectorstore, get_mmr_retriever
from src.pipeline.reranker import rerank
from src.guardrails.writer import build_cited_answer
from src.guardrails.verifier import verify_answer

vs = load_vectorstore()
retriever = get_mmr_retriever(vs)

# Demo: on-topic query goes through writer + verifier
question = "What is TechStore Plus's return window for refunds?"
mmr_docs = retriever.invoke(question)
top_docs = rerank(question, mmr_docs)

print("=== Citation-Binding Writer ===")
raw = build_cited_answer(question, top_docs)
print(raw)

print("\n=== Verifier + Decision Gate ===")
result = verify_answer(raw, top_docs)
print(f"Decision:           {result.decision}")
print(f"Claim support rate: {result.claim_support_rate:.2f}")
print(f"Contradiction rate: {result.contradiction_rate:.2f}")
print(f"Cited sources:      {result.cited_sources}")

In [ ]:
# Demo: off-topic query → no_answer guardrail
q_off = "What is the capital of France?"
mmr_off = retriever.invoke(q_off)
top_off = rerank(q_off, mmr_off) if mmr_off else []

if top_off:
    raw_off = build_cited_answer(q_off, top_off)
    result_off = verify_answer(raw_off, top_off)
else:
    from src.rag_agent import GuardrailedAnswer
    result_off = GuardrailedAnswer(
        answer="I don't have that information in our documentation.",
        decision="no_answer",
        claim_support_rate=0.0,
        contradiction_rate=0.0,
    )

print(f"Off-topic decision: {result_off.decision}")
print(f"Answer: {result_off.answer[:100]}")

---
## Component 3 — Multimodal Retrieval

Three modalities are indexed: **Text** (ChromaDB vector store), **Tables** (CSV → row Documents with `[TB:*]` citations), **Images** (caption-based with `[I:*]` citations).
Late fusion: each modality is queried independently, results are merged and deduplicated by source entity.

In [ ]:
# --- Modality 2: Table Retrieval ---
from src.multimodal.table_retriever import TableRetriever

tr = TableRetriever()
tr.load_tables()

print("=== Table: keyword query ===")
for doc in tr.retrieve("How much RAM does the Laptop Pro X1 have?"):
    print(f"  {doc.metadata['table_citation']}  {doc.page_content}")

print("\n=== Table: superlative query ===")
for doc in tr.retrieve("Which laptop has the most storage?", k=1):
    print(f"  {doc.metadata['table_citation']}  {doc.page_content}")

In [ ]:
# --- Modality 3: Image Retrieval ---
from src.multimodal.image_retriever import ImageRetriever

ir = ImageRetriever()
ir.load_images()

print("=== Image: product overview query ===")
for doc in ir.retrieve("Show me the Laptop Pro X1 overview image"):
    print(f"  {doc.metadata['image_citation']}")
    print(f"  Caption: {doc.page_content[:100]}...")

print("\n=== Image: warranty diagram ===")
for doc in ir.retrieve("warranty tiers comparison figure"):
    print(f"  {doc.metadata['image_citation']}")
    print(f"  Caption: {doc.page_content[:100]}...")

In [ ]:
# --- Late Fusion: all three modalities merged ---
from langchain_core.documents import Document

def late_fusion_retrieve(question, vs, tr, ir, text_k=3, table_k=3, image_k=2):
    """Query all three modalities and merge evidence with citation tags."""
    # Text modality
    retriever = get_mmr_retriever(vs)
    mmr_docs = retriever.invoke(question)
    text_docs = rerank(question, mmr_docs) if mmr_docs else []

    # Table modality
    try:
        table_docs = tr.retrieve(question, k=table_k)
        for d in table_docs:
            d.metadata["source"] = d.metadata.get("table_citation", "")
    except Exception:
        table_docs = []

    # Image modality
    try:
        image_docs = ir.retrieve(question, k=image_k)
        for d in image_docs:
            d.metadata["source"] = d.metadata.get("image_citation", "")
    except Exception:
        image_docs = []

    # Deduplicate by source key
    merged, seen_sources = [], set()
    for doc in text_docs + table_docs + image_docs:
        src = doc.metadata.get("source", "")
        if src not in seen_sources:
            seen_sources.add(src)
            merged.append(doc)
    return merged

question_mm = "Which laptop has the most storage and what is its warranty coverage?"
merged = late_fusion_retrieve(question_mm, vs, tr, ir)
print(f"Merged evidence ({len(merged)} docs):")
for doc in merged:
    print(f"  [{doc.metadata.get('source','')}]  {doc.page_content[:80]}")

---
## Full Pipeline — TechStoreRAGAgent

The agent orchestrates all components: MMR → cross-encoder → Graph RAG → Table → Image → Writer → Verifier → Decision Gate.

In [ ]:
from src.rag_agent import TechStoreRAGAgent

agent = TechStoreRAGAgent()

# Quick smoke test
r = agent.answer("What is the return period for a laptop refund?")
print(f"Decision: {r.decision}")
print(f"Answer:   {r.answer[:200]}")
print(f"Sources:  {r.cited_sources}")

---
## 10-Query Test Set

All 10 queries are run through the full Stop 3 pipeline. Results are collected for the comparison report below.

In [ ]:
TEST_QUERIES = [
    # (id, query, type)
    (1,  "Summarize TechStore Plus's return policy — cite at least two distinct rules with source references.",
         "multi-citation"),
    (2,  "Which warranty tiers cover accidental damage and what are the conditions? List with citations.",
         "entity + table"),
    (3,  "What is the step-by-step process to file a warranty claim? Cite the source document for each step.",
         "procedural"),
    (4,  "Who authored or compiled the product manuals in the corpus, and which version is current?",
         "provenance + table"),
    (5,  "Which products appear in both the laptop specs table and the premium warranty tier?",
         "cross-doc join"),
    (6,  "Identify the image or figure in your corpus that shows the Laptop Pro X1, and describe what it shows.",
         "image region"),
    (7,  "As of the current corpus, how much RAM and storage does the Laptop Pro X1 have?",
         "numeric grounding"),
    (8,  "Does the return policy allow refunds after 30 days? Cite the relevant section.",
         "temporal / policy"),
    (9,  "List all laptop models in the corpus and their storage capacity.",
         "text + table"),
    (10, "What does the corpus say about trade-in programs — which products are eligible?",
         "general"),
]

stop3_results = []
for qid, query, qtype in TEST_QUERIES:
    print(f"\n[Q{qid}] {query[:70]}...")
    r = agent.answer(query)
    stop3_results.append({
        "id": qid,
        "query": query,
        "type": qtype,
        "answer": r.answer,
        "decision": r.decision,
        "claim_support_rate": r.claim_support_rate,
        "contradiction_rate": r.contradiction_rate,
        "cited_sources": r.cited_sources,
    })
    print(f"  decision={r.decision}  support={r.claim_support_rate:.2f}  sources={r.cited_sources}")

print("\n=== All 10 queries complete ===")

---
## Comparison Report — Stop 2 Baseline vs Stop 3

Stop 2 baseline uses: MMR (k=6, fetch_k=20, λ=0.85) → cross-encoder (top-3) → direct LLM call (no guardrails, no graph, no table/image modality).

In [ ]:
# --- Stop 2 Baseline pipeline (no guardrails, no graph, no multimodal) ---
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

def stop2_answer(question, vs):
    """Minimal Stop 2 pipeline: MMR + rerank + direct LLM (no guardrails)."""
    retriever = get_mmr_retriever(vs)
    mmr_docs = retriever.invoke(question)
    if not mmr_docs:
        return "No relevant documents found.", []
    top_docs = rerank(question, mmr_docs)
    context = "\n---\n".join(d.page_content for d in top_docs)
    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    prompt = (
        f"Answer the question using the context below.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    resp = llm.invoke([HumanMessage(content=prompt)])
    sources = [d.metadata.get("source", "") for d in top_docs]
    return resp.content.strip(), sources

baseline_results = []
for qid, query, qtype in TEST_QUERIES:
    print(f"[Q{qid}] baseline...")
    ans, srcs = stop2_answer(query, vs)
    baseline_results.append({"id": qid, "answer": ans, "sources": srcs})

print("Baseline complete.")

In [ ]:
import pandas as pd

rows = []
for b, s3 in zip(baseline_results, stop3_results):
    rows.append({
        "#":             s3["id"],
        "Type":          s3["type"],
        "Stop 2 Answer": b["answer"][:150] + "...",
        "Stop 3 Answer": s3["answer"][:150] + "...",
        "Evidence Keys": ", ".join(s3["cited_sources"][:3]),
        "claim_support": f"{s3['claim_support_rate']:.2f}",
        "decision":      s3["decision"],
    })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 80)
df

---
## Metrics Summary

Per-query and aggregate metrics for Stop 3.

In [ ]:
import statistics

support_rates = [r["claim_support_rate"] for r in stop3_results]
contradiction_rates = [r["contradiction_rate"] for r in stop3_results]
decisions = [r["decision"] for r in stop3_results]

# Citation density = fraction of answers that have at least one cited source
citation_density = sum(1 for r in stop3_results if r["cited_sources"]) / len(stop3_results)

# Identify graph citations ([G:...]) and table citations ([TB:...])
graph_queries  = [r for r in stop3_results if any(s.startswith("[G:")  for s in r["cited_sources"])]
table_queries  = [r for r in stop3_results if any("[TB:" in s         for s in r["cited_sources"])]
image_queries  = [r for r in stop3_results if any(s.startswith("[I:")  for s in r["cited_sources"])]

print("=" * 60)
print("STOP 3 METRICS SUMMARY")
print("=" * 60)
print(f"Avg claim support rate :  {statistics.mean(support_rates):.3f}  (target ≥ 0.85)")
print(f"Avg contradiction rate :  {statistics.mean(contradiction_rates):.3f}  (target ≈ 0)")
print(f"Citation density        :  {citation_density:.2f}  (target ≥ 0.3)")
print()
print(f"Decision distribution:")
for d in ["answer", "answer_with_disclaimer", "extractive", "no_answer"]:
    count = decisions.count(d)
    print(f"  {d:30s}: {count}/10")
print()
print(f"Queries with Graph citations  : {len(graph_queries)}/10")
print(f"Queries with Table citations  : {len(table_queries)}/10")
print(f"Queries with Image citations  : {len(image_queries)}/10")
print("=" * 60)

In [ ]:
# Per-query metrics table
metrics_rows = []
for r in stop3_results:
    has_graph = any(s.startswith("[G:") for s in r["cited_sources"])
    has_table = any("[TB:"           in s for s in r["cited_sources"])
    has_image = any(s.startswith("[I:") for s in r["cited_sources"])
    metrics_rows.append({
        "Q#":            r["id"],
        "type":          r["type"],
        "decision":      r["decision"],
        "support":       f"{r['claim_support_rate']:.2f}",
        "contradiction": f"{r['contradiction_rate']:.2f}",
        "[G:]": "✓" if has_graph else "",
        "[TB:": "✓" if has_table else "",
        "[I:]": "✓" if has_image else "",
        "num_sources":   len(r["cited_sources"]),
    })

pd.DataFrame(metrics_rows)

---
## Analysis: What Improved and What Regressed

### What Stop 3 improved over Stop 2 baseline

1. **Citation binding** — every Stop 3 sentence ends with a `[source]` key, making answers verifiable. Stop 2 answers had no citations.

2. **Hallucination prevention** — the decision gate correctly returns `no_answer` for off-topic queries. Stop 2 would generate plausible-sounding but ungrounded answers for out-of-corpus questions.

3. **Numeric grounding** — table queries (`Q7`, `Q9`) now retrieve exact values from `laptop_specs.csv` and `warranty_tiers.csv` with `[TB:*]` citations rather than relying on embedding recall over prose.

4. **Entity relationship traversal** — queries like `Q2` (warranty tiers + damage coverage) and `Q5` (cross-doc join) benefit from the knowledge graph surfacing relationships that span multiple documents.

5. **Image evidence** — `Q6` (identify figure showing Laptop Pro X1) now returns a grounded `[I:*]` citation with descriptive caption.

### Known limitations / potential regressions

1. **Latency** — Stop 3 makes additional LLM calls per query (triple extraction on cold start, claim decomposition per answer). Cold start is ~2-3× slower than Stop 2.

2. **Triple extraction quality** — `gpt-4.1-mini` may miss low-frequency relations or hallucinate relations not present in the text. The `DEFAULT_RELATION_ALLOWLIST` mitigates but does not eliminate noise.

3. **Image retrieval is caption-based** — no actual OCR or VLM is used; image content is described by human-authored captions. A real image would require OCR/VLM for full accuracy.

4. **Knowledge graph is in-memory** — rebuilt on every cold start. For large corpora, persistence (JSON or Neo4j) would be required.

### Next steps

- Persist the knowledge graph to JSON/Neo4j to eliminate rebuild overhead
- Add VLM-based image captioning (GPT-4o vision) for real visual grounding
- Implement graph-first routing for entity-dense questions (M3 LangGraph)
- Add confidence scores to triples and filter by confidence threshold

---
## Mandatory Test Cases

Run the three graded test cases to confirm Stop 3 is complete.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_mandatory_cases.py", "-v", "-s"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)